# ResGRU-UNet — PM2.5 Forecasting

**Architecture upgrades over AFNO (0.85):**
1. **U-Net with Skip Connections** — passes high-frequency spatial detail (local spikes) directly to output, bypassing bottleneck blur
2. **ConvGRU Bottleneck** — learns how pollution plumes evolve over the 16-hour forecast horizon
3. **Log-Space Training** — `log1p(target)` normalises extreme episode values, prevents gradient explosion
4. **Episode-Weighted Huber Loss** — episodic pixels weighted **10x** to force the model to capture spikes
5. **Persistence Residual** — predicts delta from last known frame instead of absolute values
6. **Diurnal + Wind Features** — sin/cos hour-of-day + wind speed added as input channels

## Imports

In [1]:
import os
import gc
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

## Configuration

In [2]:
class Config:
    IS_KAGGLE = os.path.exists('/kaggle/input')
    if IS_KAGGLE:
        DATA_ROOT = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
        WORKING_DIR = '/kaggle/working'
    else:
        DATA_ROOT = './aisehack-theme-2'
        WORKING_DIR = '.'

    RAW_DIR = os.path.join(DATA_ROOT, 'raw')
    TEST_DIR = os.path.join(DATA_ROOT, 'test_in')

    MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']

    FEATURES = [
        'cpm25', 'q2', 't2', 'u10', 'v10', 'pblh',
        'psfc', 'swdown', 'rain', 'PM25', 'NOx', 'SO2', 'NH3',
    ]
    NUM_FEATURES = len(FEATURES)
    # +3 engineered: wind_speed, hour_sin, hour_cos
    NUM_INPUT_CHANNELS = NUM_FEATURES + 3

    MAX_SCALE_FEATURES = {'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio'}
    LOG_TRANSFORM_FEATURES = {'rain'}

    INPUT_HOURS = 10
    OUTPUT_HOURS = 16
    WINDOW_SIZE = INPUT_HOURS + OUTPUT_HOURS  # 26

    H = 140
    W = 124

    STRIDE = 2
    BATCH_SIZE = 4
    VAL_SPLIT = 0.1
    EPOCHS = 30
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = 8

    ENC_CHANNELS = [64, 128, 256]
    GRU_HIDDEN = 256
    DEC_CHANNELS = [128, 64, 32]
    DROP_RATE = 0.1

    EPISODE_STD_THRESHOLD = 2.0
    EPISODE_MIN_CPM = 1.0
    EPISODE_WEIGHT = 10.0

    INFERENCE_BATCH_SIZE = 4
    NUM_TEST_SAMPLES = 218
    SEED = 42


def seed_everything(seed=Config.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
GPU Memory: 15.6 GB


## Normalization Statistics

In [3]:
class NormStats:
    def __init__(self):
        self.mean = {}
        self.std = {}

    def compute_from_raw(self, raw_dir, months, features):
        print("Computing normalization statistics...")
        for feat in features:
            is_max_scale = feat in Config.MAX_SCALE_FEATURES
            if is_max_scale:
                global_max = 0.0
                for month in months:
                    path = os.path.join(raw_dir, month, f'{feat}.npy')
                    data = np.load(path, mmap_mode='r')
                    global_max = max(global_max, float(np.max(np.abs(data))))
                self.mean[feat] = np.zeros((Config.H, Config.W), dtype=np.float32)
                self.std[feat] = np.full((Config.H, Config.W), max(global_max, 1e-15), dtype=np.float32)
                print(f"  {feat} [max-scale]: global_max={global_max:.2e}")
            elif feat in Config.LOG_TRANSFORM_FEATURES:
                running_sum = 0.0; running_sq = 0.0; total_count = 0
                for month in months:
                    path = os.path.join(raw_dir, month, f'{feat}.npy')
                    data = np.load(path, mmap_mode='r')
                    for t in range(data.shape[0]):
                        x = np.log1p(data[t].astype(np.float64))
                        running_sum += x.sum(); running_sq += (x**2).sum(); total_count += x.size
                global_mean = running_sum / total_count
                global_std = max(np.sqrt(running_sq/total_count - global_mean**2), 1e-6)
                self.mean[feat] = np.full((Config.H, Config.W), global_mean, dtype=np.float32)
                self.std[feat] = np.full((Config.H, Config.W), global_std, dtype=np.float32)
                print(f"  {feat} [log1p+zscore]: mean={global_mean:.4f}, std={global_std:.4f}")
            else:
                count = 0
                mean_acc = np.zeros((Config.H, Config.W), dtype=np.float64)
                m2_acc = np.zeros((Config.H, Config.W), dtype=np.float64)
                for month in months:
                    path = os.path.join(raw_dir, month, f'{feat}.npy')
                    data = np.load(path, mmap_mode='r')
                    for t in range(data.shape[0]):
                        x = data[t].astype(np.float64)
                        count += 1
                        delta = x - mean_acc
                        mean_acc += delta / count
                        m2_acc += delta * (x - mean_acc)
                variance = m2_acc / max(count - 1, 1)
                self.mean[feat] = mean_acc.astype(np.float32)
                self.std[feat] = np.maximum(np.sqrt(variance).astype(np.float32), 1e-6)
                print(f"  {feat}: mean=[{self.mean[feat].min():.2f}, {self.mean[feat].max():.2f}]")

    def normalize(self, data, feat_name):
        if feat_name in Config.LOG_TRANSFORM_FEATURES:
            data = np.log1p(data)
        return (data - self.mean[feat_name]) / self.std[feat_name]

    def save(self, path):
        np.savez(path, features=list(self.mean.keys()),
                 **{f'mean_{k}': v for k, v in self.mean.items()},
                 **{f'std_{k}': v for k, v in self.std.items()})

    def load(self, path):
        d = np.load(path)
        for feat in list(d['features']):
            self.mean[feat] = d[f'mean_{feat}']
            self.std[feat] = d[f'std_{feat}']


cfg = Config
stats_path = os.path.join(cfg.WORKING_DIR, 'norm_stats_resgru.npz')
norm_stats = NormStats()

if os.path.exists(stats_path):
    print("Loading cached normalization stats...")
    norm_stats.load(stats_path)
else:
    norm_stats.compute_from_raw(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES)
    norm_stats.save(stats_path)
    print("Stats saved.")
gc.collect()

Computing normalization statistics...
  cpm25: mean=[2.51, 274.26]
  q2: mean=[0.00, 0.02]
  t2: mean=[255.68, 303.37]
  u10: mean=[-2.61, 7.09]
  v10: mean=[-3.51, 7.61]
  pblh: mean=[262.77, 1514.86]
  psfc: mean=[49109.23, 101124.63]
  swdown: mean=[129.73, 289.67]
  rain [log1p+zscore]: mean=0.0442, std=0.2089
  PM25 [max-scale]: global_max=1.43e-07
  NOx [max-scale]: global_max=7.98e-08
  SO2 [max-scale]: global_max=4.33e-08
  NH3 [max-scale]: global_max=2.09e-08
Stats saved.


92

## Dataset Classes (Log-Space + Diurnal/Wind Features)

In [4]:
def compute_engineered_features(u10, v10, hour_indices):
    """Wind speed + diurnal sin/cos from normalized u10/v10."""
    wind_speed = np.sqrt(u10**2 + v10**2).astype(np.float32)
    T, H, W = u10.shape
    hours = hour_indices.astype(np.float32)
    hour_sin = np.sin(2 * np.pi * hours / 24.0)[:, None, None] * np.ones((1, H, W), dtype=np.float32)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)[:, None, None] * np.ones((1, H, W), dtype=np.float32)
    return wind_speed, hour_sin, hour_cos


class PM25TrainDataset(Dataset):
    def __init__(self, raw_dir, months, features, norm_stats, stride=Config.STRIDE):
        self.features = features
        self.norm_stats = norm_stats
        self.samples = []
        self.data_paths = {}

        for m_idx, month in enumerate(months):
            path = os.path.join(raw_dir, month, 'cpm25.npy')
            T = np.load(path, mmap_mode='r').shape[0]
            n_samples = (T - Config.WINDOW_SIZE) // stride + 1
            for i in range(n_samples):
                self.samples.append((m_idx, i * stride))
            for feat in features:
                self.data_paths[(m_idx, feat)] = os.path.join(raw_dir, month, f'{feat}.npy')

        print(f"  TrainDataset: {len(self.samples)} samples, stride={stride}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        m_idx, start = self.samples[idx]
        input_list = []
        u10_norm = v10_norm = None

        for feat in self.features:
            raw = np.load(self.data_paths[(m_idx, feat)], mmap_mode='r')
            chunk = self.norm_stats.normalize(raw[start:start+Config.INPUT_HOURS].astype(np.float32), feat)
            input_list.append(chunk)
            if feat == 'u10': u10_norm = chunk
            elif feat == 'v10': v10_norm = chunk

        hour_idx = np.arange(start, start + Config.INPUT_HOURS) % 24
        ws, hs, hc = compute_engineered_features(u10_norm, v10_norm, hour_idx)
        input_list.extend([ws, hs, hc])
        inputs = np.stack(input_list, axis=0)  # (C, 10, H, W)

        cpm_raw = np.load(self.data_paths[(m_idx, 'cpm25')], mmap_mode='r')
        target_raw = cpm_raw[start+Config.INPUT_HOURS:start+Config.WINDOW_SIZE].astype(np.float32)
        target_log = np.log1p(target_raw)
        last_frame_log = np.log1p(cpm_raw[start+Config.INPUT_HOURS-1].astype(np.float32))
        context_raw = cpm_raw[start:start+Config.INPUT_HOURS].astype(np.float32)

        return (torch.from_numpy(inputs),
                torch.from_numpy(target_log),
                torch.from_numpy(target_raw),
                torch.from_numpy(last_frame_log),
                torch.from_numpy(context_raw))


class PM25TestDataset(Dataset):
    def __init__(self, test_dir, features, norm_stats):
        self.features = features
        self.norm_stats = norm_stats
        self.data = {feat: np.load(os.path.join(test_dir, f'{feat}.npy'), mmap_mode='c') for feat in features}
        self.num_samples = self.data[features[0]].shape[0]
        print(f"  TestDataset: {self.num_samples} samples (mmap)")

    def __len__(self): return self.num_samples

    def __getitem__(self, idx):
        input_list = []
        u10_norm = v10_norm = None
        for feat in self.features:
            chunk = self.norm_stats.normalize(self.data[feat][idx].astype(np.float32), feat)
            input_list.append(chunk)
            if feat == 'u10': u10_norm = chunk
            elif feat == 'v10': v10_norm = chunk

        hour_idx = np.arange(Config.INPUT_HOURS).astype(np.float32)
        ws, hs, hc = compute_engineered_features(u10_norm, v10_norm, hour_idx)
        input_list.extend([ws, hs, hc])
        inputs = np.stack(input_list, axis=0)

        last_frame_log = np.log1p(self.data['cpm25'][idx][-1].astype(np.float32))
        return torch.from_numpy(inputs), torch.from_numpy(last_frame_log)


full_dataset = PM25TrainDataset(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES, norm_stats, stride=cfg.STRIDE)
n_val = int(len(full_dataset) * cfg.VAL_SPLIT)
n_train = len(full_dataset) - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(cfg.SEED)
)
print(f"Train: {n_train}, Val: {n_val}")

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True, persistent_workers=True)

  TrainDataset: 1416 samples, stride=2
Train: 1275, Val: 141


## ResGRU-UNet Architecture

In [5]:
class ConvBlock(nn.Module):
    """Two 3x3 convs with GroupNorm + GELU + residual."""
    def __init__(self, in_ch, out_ch, drop_rate=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.drop = nn.Dropout2d(drop_rate) if drop_rate > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        r = self.skip(x)
        x = F.gelu(self.gn1(self.conv1(x)))
        x = self.drop(x)
        x = F.gelu(self.gn2(self.conv2(x)))
        return x + r


class ConvGRUCell(nn.Module):
    """Convolutional GRU cell — learns temporal evolution on 2D spatial grids."""
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        p = kernel_size // 2
        self.conv_gates = nn.Conv2d(input_dim + hidden_dim, 2*hidden_dim, kernel_size, padding=p)
        self.conv_candidate = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=p)
        self.gn_gates = nn.GroupNorm(min(32, 2*hidden_dim), 2*hidden_dim)
        self.gn_cand = nn.GroupNorm(min(32, hidden_dim), hidden_dim)

    def forward(self, x, h):
        gates = torch.sigmoid(self.gn_gates(self.conv_gates(torch.cat([x, h], 1))))
        r, u = gates.chunk(2, dim=1)
        cand = torch.tanh(self.gn_cand(self.conv_candidate(torch.cat([x, r*h], 1))))
        return (1 - u) * h + u * cand


class ResGRUUNet(nn.Module):
    """
    3-level U-Net with ConvGRU bottleneck for spatiotemporal PM2.5 forecasting.

    Input:  (B, C, T_in, H, W)  — C = NUM_INPUT_CHANNELS (16 channels)
    Output: (B, T_out, H, W)    — log1p(cpm25) per output hour
    """
    def __init__(self, cfg=Config):
        super().__init__()
        self.cfg = cfg
        C_in = cfg.NUM_INPUT_CHANNELS * cfg.INPUT_HOURS
        enc_ch = cfg.ENC_CHANNELS   # [64, 128, 256]
        dec_ch = cfg.DEC_CHANNELS   # [128, 64, 32]
        drop = cfg.DROP_RATE

        # Encoder
        self.enc0 = ConvBlock(C_in, enc_ch[0], drop)
        self.pool0 = nn.MaxPool2d(2)

        self.enc1 = ConvBlock(enc_ch[0], enc_ch[1], drop)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ConvBlock(enc_ch[1], enc_ch[2], drop)
        self.pool2 = nn.AvgPool2d(2, ceil_mode=True)

        # ConvGRU bottleneck
        self.bottleneck = ConvBlock(enc_ch[2], cfg.GRU_HIDDEN, drop)
        self.convgru = ConvGRUCell(cfg.GRU_HIDDEN, cfg.GRU_HIDDEN)
        self.gru_proj = nn.Conv2d(cfg.GRU_HIDDEN, enc_ch[2], 1, bias=False)

        # Decoder
        self.up2 = nn.ConvTranspose2d(enc_ch[2], dec_ch[0], 2, stride=2)
        self.dec2 = ConvBlock(dec_ch[0] + enc_ch[2], dec_ch[0], drop)

        self.up1 = nn.ConvTranspose2d(dec_ch[0], dec_ch[1], 2, stride=2)
        self.dec1 = ConvBlock(dec_ch[1] + enc_ch[1], dec_ch[1], drop)

        self.up0 = nn.ConvTranspose2d(dec_ch[1], dec_ch[2], 2, stride=2)
        self.dec0 = ConvBlock(dec_ch[2] + enc_ch[0], dec_ch[2], drop)

        # Delta head (persistence residual)
        self.head = nn.Sequential(
            nn.Conv2d(dec_ch[2], dec_ch[2], 3, padding=1, bias=False),
            nn.GroupNorm(min(32, dec_ch[2]), dec_ch[2]),
            nn.GELU(),
            nn.Conv2d(dec_ch[2], 1, 1),
        )
        # Init head to zero for stable start
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def _crop(self, x, ref):
        """Trim x to match ref's spatial dims after upsampling odd sizes."""
        return x[:, :, :ref.shape[2], :ref.shape[3]]

    def forward(self, x, last_frame_log):
        B, C, T, H, W = x.shape
        x = x.reshape(B, C*T, H, W)

        e0 = self.enc0(x)
        e1 = self.enc1(self.pool0(e0))
        e2 = self.enc2(self.pool1(e1))
        bn = self.bottleneck(self.pool2(e2))

        h = bn
        outputs = []
        for _ in range(self.cfg.OUTPUT_HOURS):
            h = self.convgru(bn, h)
            d = self.gru_proj(h)

            d = self._crop(self.up2(d), e2)
            d = self.dec2(torch.cat([d, e2], 1))

            d = self._crop(self.up1(d), e1)
            d = self.dec1(torch.cat([d, e1], 1))

            d = self._crop(self.up0(d), e0)
            d = self.dec0(torch.cat([d, e0], 1))

            outputs.append(self.head(d).squeeze(1))

        deltas = torch.stack(outputs, dim=1)                    # (B, T_out, H, W)
        return last_frame_log.unsqueeze(1) + deltas             # persistence residual


model = ResGRUUNet(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,} ({n_params/1e6:.1f}M)")

# Shape sanity check
dummy_x = torch.randn(1, cfg.NUM_INPUT_CHANNELS, cfg.INPUT_HOURS, cfg.H, cfg.W).to(device)
dummy_lf = torch.randn(1, cfg.H, cfg.W).to(device)
with autocast(dtype=torch.float16):
    out = model(dummy_x, dummy_lf)
print(f"Input: {dummy_x.shape}  Last frame: {dummy_lf.shape}  Output: {out.shape}")
assert out.shape == (1, cfg.OUTPUT_HOURS, cfg.H, cfg.W)
del dummy_x, dummy_lf, out
torch.cuda.empty_cache()

Parameters: 7,096,513 (7.1M)


/tmp/ipykernel_24/1656378427.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


Input: torch.Size([1, 16, 10, 140, 124])  Last frame: torch.Size([1, 140, 124])  Output: torch.Size([1, 16, 140, 124])


## Episode-Weighted Huber Loss (Log-Space)

In [6]:
class EpisodeWeightedHuberLoss(nn.Module):
    """
    Weighted SmoothL1 loss in log1p space.
    Episodic pixels (target > baseline + 2σ AND > 1 μg/m³) weighted 10x.
    Plus Pearson correlation loss on episodic pixels.
    """
    def __init__(self, ep_weight=Config.EPISODE_WEIGHT,
                 threshold=Config.EPISODE_STD_THRESHOLD,
                 min_cpm=Config.EPISODE_MIN_CPM, beta=1.0, eps=1e-8):
        super().__init__()
        self.ep_weight = ep_weight
        self.threshold = threshold
        self.min_cpm = min_cpm
        self.beta = beta
        self.eps = eps

    def forward(self, pred_log, target_log, target_raw, last_frame_log, context_cpm_raw):
        # Episode mask from raw values using context baseline
        baseline_mean = context_cpm_raw.mean(dim=1, keepdim=True)
        baseline_std  = context_cpm_raw.std(dim=1, keepdim=True) + self.eps
        episode_mask = ((target_raw > baseline_mean + self.threshold * baseline_std) &
                        (target_raw > self.min_cpm)).float()

        weights = 1.0 + (self.ep_weight - 1.0) * episode_mask
        huber = F.smooth_l1_loss(pred_log, target_log, beta=self.beta, reduction='none')
        weighted_huber = (huber * weights).mean()

        # Correlation loss on episodic pixels
        pred_raw = F.relu(torch.expm1(pred_log.clamp(max=15)))
        corr_loss = self._corr_loss(pred_raw, target_raw, episode_mask)

        total = weighted_huber + 0.3 * corr_loss

        with torch.no_grad():
            numer = torch.abs(pred_raw - target_raw)
            denom = (torch.abs(pred_raw) + torch.abs(target_raw)) * 0.5 + self.eps
            smape = (numer / denom).mean().item()
            n_ep = episode_mask.sum().item()
            ep_smape = ((numer/denom) * episode_mask).sum().item() / max(n_ep, 1)

        return total, {
            'huber': weighted_huber.item(),
            'corr_loss': corr_loss.item(),
            'global_smape': smape,
            'episode_smape': ep_smape,
            'n_episodes': n_ep,
        }

    def _corr_loss(self, pred_raw, target_raw, mask):
        corr_sum = 0.0; valid_t = 0
        for t in range(pred_raw.shape[1]):
            ep_idx = mask[:, t].reshape(-1) > 0.5
            if ep_idx.sum() < 10: continue
            p = pred_raw[:, t].reshape(-1)[ep_idx]
            y = target_raw[:, t].reshape(-1)[ep_idx]
            p_c = p - p.mean(); y_c = y - y.mean()
            cov = (p_c * y_c).sum()
            corr = cov / (torch.sqrt((p_c**2).sum() + self.eps) * torch.sqrt((y_c**2).sum() + self.eps))
            corr_sum += corr; valid_t += 1
        if valid_t == 0: return torch.tensor(0.0, device=pred_raw.device)
        return 1.0 - corr_sum / valid_t

criterion = EpisodeWeightedHuberLoss()
print("Loss function ready.")

Loss function ready.


## Training Loop with OneCycleLR

In [7]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss = 0.0; metrics_acc = {}; n_batches = 0

    for i, (inputs, target_log, target_raw, last_frame_log, context_raw) in enumerate(loader):
        inputs        = inputs.to(device, non_blocking=True)
        target_log    = target_log.to(device, non_blocking=True)
        target_raw    = target_raw.to(device, non_blocking=True)
        last_frame_log= last_frame_log.to(device, non_blocking=True)
        context_raw   = context_raw.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            pred_log = model(inputs, last_frame_log)
            loss, metrics = criterion(pred_log.float(), target_log, target_raw,
                                      last_frame_log, context_raw)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        for k, v in metrics.items(): metrics_acc[k] = metrics_acc.get(k, 0) + v
        n_batches += 1

        if i % 50 == 0:
            print(f"  [{i}/{len(loader)}] loss={loss.item():.4f} "
                  f"smape={metrics['global_smape']:.4f} lr={optimizer.param_groups[0]['lr']:.2e}")

    avg_loss = total_loss / max(n_batches, 1)
    for k in metrics_acc: metrics_acc[k] /= max(n_batches, 1)
    return avg_loss, metrics_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0; metrics_acc = {}; n_batches = 0

    for inputs, target_log, target_raw, last_frame_log, context_raw in loader:
        inputs        = inputs.to(device, non_blocking=True)
        target_log    = target_log.to(device, non_blocking=True)
        target_raw    = target_raw.to(device, non_blocking=True)
        last_frame_log= last_frame_log.to(device, non_blocking=True)
        context_raw   = context_raw.to(device, non_blocking=True)

        with autocast(dtype=torch.float16):
            pred_log = model(inputs, last_frame_log)
            loss, metrics = criterion(pred_log.float(), target_log, target_raw,
                                      last_frame_log, context_raw)

        total_loss += loss.item()
        for k, v in metrics.items(): metrics_acc[k] = metrics_acc.get(k, 0) + v
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    for k in metrics_acc: metrics_acc[k] /= max(n_batches, 1)
    return avg_loss, metrics_acc


# ---- Train ----
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=cfg.LR, epochs=cfg.EPOCHS,
    steps_per_epoch=len(train_loader), pct_start=0.1,
    anneal_strategy='cos', div_factor=25, final_div_factor=1e4,
)
scaler = GradScaler()

best_val_loss = float('inf')
patience_counter = 0
ckpt_path = os.path.join(cfg.WORKING_DIR, 'best_resgru_unet.pt')

print(f"OneCycleLR: max_lr={cfg.LR}, epochs={cfg.EPOCHS}")
print("Starting training...")

for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()
    train_loss, train_m = train_one_epoch(model, train_loader, criterion,
                                          optimizer, scheduler, scaler, device, epoch)
    val_loss, val_m = validate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    print(f"\nEpoch {epoch}/{cfg.EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train: loss={train_loss:.4f} smape={train_m.get('global_smape',0):.4f} "
          f"huber={train_m.get('huber',0):.4f}")
    print(f"  Val:   loss={val_loss:.4f}  smape={val_m.get('global_smape',0):.4f} "
          f"huber={val_m.get('huber',0):.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_loss': val_loss}, ckpt_path)
        print(f"  -> Saved best model (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= cfg.PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

OneCycleLR: max_lr=0.001, epochs=30
Starting training...


/tmp/ipykernel_24/1868865585.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_24/1868865585.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  [0/318] loss=0.3978 smape=0.3243 lr=4.00e-05
  [50/318] loss=0.4190 smape=0.2874 lr=4.68e-05
  [100/318] loss=0.3969 smape=0.3208 lr=6.64e-05
  [150/318] loss=0.2116 smape=0.3072 lr=9.82e-05
  [200/318] loss=0.2161 smape=0.3159 lr=1.42e-04
  [250/318] loss=0.1646 smape=0.3308 lr=1.95e-04
  [300/318] loss=0.2520 smape=0.3160 lr=2.58e-04


/tmp/ipykernel_24/1868865585.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):



Epoch 1/30 (140s)
  Train: loss=0.2748 smape=0.3076 huber=0.2532
  Val:   loss=0.2169  smape=0.3319 huber=0.1933
  -> Saved best model (val_loss=0.2169)
  [0/318] loss=0.2143 smape=0.3330 lr=2.82e-04
  [50/318] loss=0.1726 smape=0.3278 lr=3.53e-04
  [100/318] loss=0.2037 smape=0.3427 lr=4.30e-04
  [150/318] loss=0.1641 smape=0.3039 lr=5.08e-04
  [200/318] loss=0.1839 smape=0.3168 lr=5.87e-04
  [250/318] loss=0.2166 smape=0.3378 lr=6.64e-04
  [300/318] loss=0.1824 smape=0.3126 lr=7.37e-04

Epoch 2/30 (118s)
  Train: loss=0.1980 smape=0.3259 huber=0.1798
  Val:   loss=0.1886  smape=0.3100 huber=0.1672
  -> Saved best model (val_loss=0.1886)
  [0/318] loss=0.1694 smape=0.3152 lr=7.62e-04
  [50/318] loss=0.1662 smape=0.3409 lr=8.27e-04
  [100/318] loss=0.1509 smape=0.3406 lr=8.83e-04
  [150/318] loss=0.1796 smape=0.3113 lr=9.30e-04
  [200/318] loss=0.1733 smape=0.3005 lr=9.65e-04
  [250/318] loss=0.1603 smape=0.3397 lr=9.89e-04
  [300/318] loss=0.1856 smape=0.3180 lr=9.99e-04

Epoch 3/30 

## OOM-Proof Inference (Log-Space → Raw)

In [8]:
# Load best checkpoint
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

del train_loader, val_loader, train_ds, val_ds, full_dataset
gc.collect()
torch.cuda.empty_cache()

# Inference
model.eval()
output_path = os.path.join(cfg.WORKING_DIR, 'preds.npy')
preds = np.zeros((cfg.NUM_TEST_SAMPLES, cfg.H, cfg.W, cfg.OUTPUT_HOURS), dtype=np.float32)

test_ds = PM25TestDataset(cfg.TEST_DIR, cfg.FEATURES, norm_stats)
test_loader = DataLoader(test_ds, batch_size=cfg.INFERENCE_BATCH_SIZE,
                         shuffle=False, num_workers=0, pin_memory=True)

sample_idx = 0
with torch.no_grad():
    for batch_inputs, batch_last_frame in test_loader:
        batch_inputs     = batch_inputs.to(device, non_blocking=True)
        batch_last_frame = batch_last_frame.to(device, non_blocking=True)
        bs = batch_inputs.shape[0]

        with autocast(dtype=torch.float16):
            pred_log = model(batch_inputs, batch_last_frame).float()

        # expm1 to go from log-space back to raw PM2.5
        pred_raw = F.relu(torch.expm1(pred_log.clamp(max=15)))

        pred_np = pred_raw.cpu().numpy()
        pred_np = np.transpose(pred_np, (0, 2, 3, 1))  # (B, H, W, 16)
        preds[sample_idx:sample_idx+bs] = pred_np
        sample_idx += bs

        if sample_idx % 50 == 0 or sample_idx == cfg.NUM_TEST_SAMPLES:
            print(f"  Inference: {sample_idx}/{cfg.NUM_TEST_SAMPLES}")

        del batch_inputs, batch_last_frame, pred_log, pred_raw
        torch.cuda.empty_cache()

np.save(output_path, preds)
print(f"\nSaved to {output_path}")
print(f"Shape:      {preds.shape}")
print(f"Expected:   (218, 140, 124, 16)")
print(f"Range:      [{preds.min():.2f}, {preds.max():.2f}]")
print(f"Mean:       {preds.mean():.2f}")
assert preds.shape == (218, 140, 124, 16), f"Shape mismatch: {preds.shape}"
print("\n✓ Submission file is ready!")

Loaded best model from epoch 30 (val_loss=0.0466)
  TestDataset: 218 samples (mmap)


/tmp/ipykernel_24/1723741658.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


  Inference: 100/218
  Inference: 200/218
  Inference: 218/218

Saved to /kaggle/working/preds.npy
Shape:      (218, 140, 124, 16)
Expected:   (218, 140, 124, 16)
Range:      [0.00, 1358.56]
Mean:       38.55

✓ Submission file is ready!
